# Piso de rodadas: caminhos B, C, D e F com 10x os dados

Gerado por `scripts/reduced_rounds/kaggle_publicar.py`. Para uso manual: suba
`build/kaggle_piso` como Dataset, acelerador **GPU T4** (não P100), e rode com
*Save & Run All*. Para retomar, adicione a saída da versão anterior como Input.


In [ ]:
ALGOS = ['ascon']
LIMITE_HORAS = 11.0
EXTRA = []   # ['--sem-meta'] deixa o Meta_F de fora

import glob, os, shutil, subprocess, sys, time
T0 = time.time()
codigo = os.path.dirname(glob.glob('/kaggle/input/**/codigo/scripts', recursive=True)[0])
DADOS = os.path.dirname(glob.glob('/kaggle/input/**/ascon_texto_k3000.npz', recursive=True)[0])
print('código em', codigo, '| dados em', DADOS)
print(sorted(os.listdir(DADOS)))
subprocess.run(['nvidia-smi'])
import torch
assert torch.cuda.is_available(), 'sem GPU: ligar o acelerador T4'
gpu = torch.cuda.get_device_name(0)
assert 'P100' not in gpu, f'{gpu}: o PyTorch atual não roda na P100, trocar para T4'
print('GPU:', gpu, '| torch', torch.__version__)

In [ ]:
SAIDA = '/kaggle/working/piso'
for algo in ALGOS:
    ant = [p for p in glob.glob(f'/kaggle/input/**/{algo}_floor/feitos.txt', recursive=True)
           if not p.startswith(os.path.dirname(codigo))]
    if ant:
        origem = os.path.dirname(ant[0])
        shutil.copytree(origem, os.path.join(SAIDA, f'{algo}_floor'), dirs_exist_ok=True)
        print('retomando', algo, 'de', origem)

In [ ]:
runner = os.path.join(codigo, 'scripts', 'reduced_rounds', 'kaggle_piso.py')
for algo in ALGOS:
    resta = LIMITE_HORAS - (time.time() - T0) / 3600
    if resta < 0.5:
        print('sem tempo para', algo); continue
    # A saída do runner passa linha a linha pelo print do notebook: cada
    # RESULTADO fica no log no instante em que é calculado.
    proc = subprocess.Popen([sys.executable, '-u', runner, '--algo', algo, '--dados', DADOS,
                             '--saida', SAIDA, '--limite-horas', f'{resta:.2f}', *EXTRA],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linha in proc.stdout:
        print(linha, end='', flush=True)
    print('runner terminou com código', proc.wait(), flush=True)

In [ ]:
for algo in ALGOS:
    for nome in ('piso.csv', 'feitos.txt'):
        p = os.path.join(SAIDA, f'{algo}_floor', nome)
        if os.path.exists(p):
            print(f'==== {algo} {nome}'); print(open(p, encoding='utf-8').read())